# Garmin Daily Ingestion -- Direct Delta Write

Pull health data from Garmin Connect and write directly to the `wearables_zerobus`
bronze Delta table using Spark SQL with `parse_json()`. This bypasses ZeroBus
and is the simplest path for testing.

**Prerequisites:**
1. Deploy `zeroBus/dbxW_zerobus_infra` bundle first (creates schema, table, service principal).
2. Push Garmin tokens to the shared secret scope:
   ```
   databricks secrets put-secret dbxw_zerobus_credentials garmin_oauth_tokens \
     --string-value "$(cat ~/.garminconnect/garmin_tokens.json)"
   ```

In [ ]:
%pip install garminconnect>=0.2.0

In [ ]:
DEFAULT_CATALOG = ""  # Must be supplied via widget or ${var.catalog} job parameter
DEFAULT_SCHEMA = "wearables"
DEFAULT_SECRET_SCOPE = "dbxw_zerobus_credentials"
DEFAULT_DEVICE_ID = "garmin_forerunner_265"

try:
    dbutils.widgets.text("catalog", DEFAULT_CATALOG, "Catalog")
    dbutils.widgets.text("schema", DEFAULT_SCHEMA, "Schema")
    dbutils.widgets.text("secret_scope", DEFAULT_SECRET_SCOPE, "Secret Scope")
    dbutils.widgets.text("target_date", "", "Target Date (YYYY-MM-DD, blank=yesterday)")
    dbutils.widgets.text("device_id", DEFAULT_DEVICE_ID, "Device ID")

    catalog = dbutils.widgets.get("catalog")
    schema = dbutils.widgets.get("schema")
    secret_scope = dbutils.widgets.get("secret_scope")
    target_date_str = dbutils.widgets.get("target_date")
    device_id = dbutils.widgets.get("device_id")
except Exception:
    catalog = DEFAULT_CATALOG
    schema = DEFAULT_SCHEMA
    secret_scope = DEFAULT_SECRET_SCOPE
    target_date_str = ""
    device_id = DEFAULT_DEVICE_ID

spark.sql(f"USE CATALOG {catalog}")

bronze_table = f"{catalog}.{schema}.wearables_zerobus"

print(f"Catalog:      {catalog}")
print(f"Schema:       {schema}")
print(f"Bronze table: {bronze_table}")
print(f"Secret scope: {secret_scope}")
print(f"Device ID:    {device_id}")

## Determine Target Date

In [ ]:
from datetime import date, timedelta

if target_date_str and target_date_str.strip():
    target_date = date.fromisoformat(target_date_str.strip())
else:
    target_date = date.today() - timedelta(days=1)

print(f"Target date: {target_date}")

## Load Garmin Tokens and Authenticate

In [ ]:
from pathlib import Path

garmin_tokens_json = dbutils.secrets.get(scope=secret_scope, key="garmin_oauth_tokens")

tokenstore = Path.home() / ".garminconnect"
tokenstore.mkdir(parents=True, exist_ok=True)
token_path = tokenstore / "garmin_tokens.json"
token_path.write_text(garmin_tokens_json)

print(f"Garmin tokens written to {token_path}")

In [ ]:
from garminconnect import Garmin

client = Garmin()
client.login(str(tokenstore))
print("Garmin: authenticated using saved tokens")

## Extract Daily Data from Garmin Connect

Pulls all 10 API endpoints for the target date. Each is wrapped in a try/except
so a single endpoint failure does not block the rest.

In [ ]:
ds = target_date.isoformat()

EXTRACTORS = [
    ("daily_stats", "get_stats"),
    ("heart_rates", "get_heart_rates"),
    ("sleep", "get_sleep_data"),
    ("stress", "get_stress_data"),
    ("hrv", "get_hrv_data"),
    ("spo2", "get_spo2_data"),
    ("body_battery", "get_body_battery"),
    ("steps", "get_steps_data"),
    ("respiration", "get_respiration_data"),
]

raw_by_type = {}

for record_type, method_name in EXTRACTORS:
    try:
        raw_by_type[record_type] = getattr(client, method_name)(ds)
        print(f"  {record_type}: OK")
    except Exception as e:
        print(f"  {record_type}: FAILED ({e})")

try:
    raw_by_type["activities"] = client.get_activities_fordate(ds)
    print(f"  activities: OK")
except Exception as e:
    print(f"  activities: FAILED ({e})")

print(f"\nExtraction complete for {ds}: {len(raw_by_type)} categories")

## Build Bronze Rows (VARIANT Format)

Each Garmin API category becomes one row in `wearables_zerobus`.
The raw JSON is stored as VARIANT via `parse_json()`.

In [ ]:
import json
import uuid
from datetime import datetime, timezone


def to_bronze_row(raw_data, record_type, dev_id, target_ds):
    now = datetime.now(timezone.utc).isoformat()
    body = {
        "source": "garmin_connect",
        "device_id": dev_id,
        "date": target_ds,
        "data": raw_data,
    }
    headers = {
        "Content-Type": "application/json",
        "X-Platform": "garmin_connect",
        "X-Record-Type": record_type,
        "X-Device-Id": dev_id,
        "X-Upload-Timestamp": now,
    }
    return {
        "record_id": str(uuid.uuid4()),
        "ingested_at": now,
        "body_json": json.dumps(body, default=str),
        "headers_json": json.dumps(headers),
        "record_type": record_type,
    }


bronze_rows = []
for record_type, raw_data in raw_by_type.items():
    if raw_data is None:
        continue
    bronze_rows.append(to_bronze_row(raw_data, record_type, device_id, ds))

print(f"Built {len(bronze_rows)} bronze rows")
for row in bronze_rows:
    print(f"  record_type={row['record_type']}  record_id={row['record_id'][:8]}...")

## Preview Sample Data

Show the rows as a DataFrame before writing to verify the structure.

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StringType, StructField, StructType

staging_schema = StructType([
    StructField("record_id", StringType(), False),
    StructField("ingested_at", StringType(), False),
    StructField("body_json", StringType(), False),
    StructField("headers_json", StringType(), False),
    StructField("record_type", StringType(), True),
])

rows = [Row(**r) for r in bronze_rows]
staging_df = spark.createDataFrame(rows, schema=staging_schema)
print(f"Created staging DataFrame with {staging_df.count()} rows")
display(staging_df.select("record_id", "record_type", "ingested_at"))

## Write to Bronze Table (Direct Delta)

Uses `parse_json()` to convert the JSON strings into VARIANT columns
matching the `wearables_zerobus` DDL. No ZeroBus required.

In [ ]:
staging_df.createOrReplaceTempView("garmin_staging")

spark.sql(f"""
    INSERT INTO {bronze_table} (record_id, ingested_at, body, headers, record_type)
    SELECT
        record_id,
        CAST(ingested_at AS TIMESTAMP),
        parse_json(body_json),
        parse_json(headers_json),
        record_type
    FROM garmin_staging
""")

print(f"Wrote {len(bronze_rows)} rows to {bronze_table}")

## Save Refreshed Tokens Back to Secrets

The garminconnect library may have refreshed the OAuth access token during API calls.
Write the updated token file back so the next run uses the fresh token.

In [ ]:
refreshed_tokens = token_path.read_text()

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.secrets.put_secret(scope=secret_scope, key="garmin_oauth_tokens", string_value=refreshed_tokens)
print(f"Refreshed Garmin tokens saved back to scope '{secret_scope}'")

## Verify

Query the bronze table to confirm Garmin data landed correctly.
Uses VARIANT path expressions to filter by platform and extract nested fields.

In [ ]:
print(f"=== {bronze_table} ===")
print()

total_df = spark.sql(f"""
    SELECT COUNT(*) as total_rows
    FROM {bronze_table}
    WHERE headers:`X-Platform`::STRING = 'garmin_connect'
""")
display(total_df)

print("\n--- Record types ---")
types_df = spark.sql(f"""
    SELECT
        record_type,
        COUNT(*) as cnt,
        MIN(ingested_at) as earliest,
        MAX(ingested_at) as latest
    FROM {bronze_table}
    WHERE headers:`X-Platform`::STRING = 'garmin_connect'
    GROUP BY record_type
    ORDER BY cnt DESC
""")
display(types_df)

print("\n--- Sample body (first row) ---")
sample_df = spark.sql(f"""
    SELECT record_type, body:source::STRING as source, body:date::STRING as data_date
    FROM {bronze_table}
    WHERE headers:`X-Platform`::STRING = 'garmin_connect'
    LIMIT 5
""")
display(sample_df)